# ABC_GP IPM Vignette 

This notebook provides a small end-to-end example of the codebase. It runs a reduced version of the full workflow and, at each step, explains 
- **(1)** its purpose, 
- **(2)** shows the actual code that does it, and 
- **(3)** lists the files in the simulation and real case studies that implement it.


The method (described in the paper *Population-informed Joint Calibration of Vital Rates in IPMs via
Gaussian Processes and ABC*) is a pipeline that can be generally summarised as:

> **data → a GP per vital rate → summary statistic selection per vital rate → cache the GP
> posteriors → ABC calibration → use the ABC output**

Everything below runs in about a minute and writes only a few small files into a local folder
`sandbox/`. It reuses the real `simulation_case_study/` code **without editing any source file**.
Please also check `README.md` for the file-by-file overview.

**File name conventions:** 
- in the *simulation* study a trailing `2`
on a filename (e.g. `s0_gp_MCMC_sur2.py`) denotes the `D_gp` (GP-truth) scenario;
no `2` = the `D_glm` (GLM-truth) scenario. The two are otherwise identical. 
- `sep` vs `nonsep` = separate growth GPs for breeders/non-breeders vs one pooled
growth GP.

---
## Setup

Find the repo, make the real code importable, and create a `sandbox/` folder for this Vignette. The analysis code builds file paths from `os.getcwd()`, so we`chdir`
into the sandbox, that is how the Vignette reads/writes files here
instead of touching the multi-GB `true_popu/` caches of the full study.

In [ ]:
import os, sys, pickle, types, warnings, logging, io, contextlib
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
logging.getLogger("tensorflow").setLevel(logging.ERROR)

# locate the repo root (walk up until we find simulation_case_study/)
d = os.path.abspath(os.getcwd())
while d != "/" and not os.path.isdir(os.path.join(d, "simulation_case_study")):
    d = os.path.dirname(d)

REPO = d
SIM  = os.path.join(REPO, "simulation_case_study")
HERE = os.path.join(REPO, "toy_example")
SANDBOX  = os.path.join(HERE, "sandbox")
TOY_DATA = os.path.join(HERE, "toy_data")
sys.path.insert(0, SIM)  # so `import s0_fun_base` finds the real code

# the directory the real code expects under its working directory
TP = "true_popu/glm_mle_true_population"
for sub in ([TP, TP + "/MCMC/gp/sep", TP + "/MCMC/gp/nonsep"]
            + [f"true_popu/mcmc/Lm/glm/{r}" for r in ["sur","grw_f","grw_nf","grw","fec","flow"]]
            + ["ABC_details/glmsep"]):
    os.makedirs(os.path.join(SANDBOX, sub), exist_ok=True)
os.chdir(SANDBOX)
print("repo: ", REPO)
print("working: ", os.getcwd())

### Demo settings 

This vignette only provides a small example of the codebase for illustration purpose. We are doing all settings here to make it "tiny".

In [2]:
SEED = 0
N_ROWS = 40  # rows of a small data used for the vignette
N_BURNIN = 10  # MCMC warm-up steps used for the vignette (full: 5000-100000, per vital rate)
N_DRAWS = 6  # MCMC draws per vital rate used for the vignette (full: 5000), This is also the number of 'cache' files
N_LEAPFROG = 5  # leapfrog steps for HMC (a MCMC sampler)
ABC_QUANTILES = [0.8, 0.6]  # ABC tolerance schedule (we used a 'loose' setup, just for illustrative purposes)
ABC_NPARTICLES = [60, 30]  # particles required per ABC round (we used a 'loose' setup, just for illustrative purposes)
ABC_BATCH = 100  # candidates per inner pass, i.e., the number of candidates to generate before we check for acceptance

# recruitment parameters (from Table 2 in the main text)
ALPHA, BETA, RECRUIT_P = 0.706, 1.329, 0.210

---
## Step 0: the data

**Purpose.** Everything starts from a demographic dataset with one row per
individual per year-transition. The columns are `size, sizeNext, fec, flow,
surv, age, ageNext`: current and next size, a breeder indicator `fec`, flowering stalks
count `flow`, survival `surv`, and age. 

Growth is later split into **breeders vs
non-breeders**, so the growth process is conditional on reproductive status. Our toy dataset is a ~40-row subsample of the published *simulated*
population.

In [3]:
from s0_fun_base import (XY_sur_compu, XY_fec_compu, XY_flow_compu,
                         XY_grw_f_compu, XY_grw_nf_compu, XY_grw_compu)

toy_path = os.path.join(TOY_DATA, "toy_population.pkl")
if os.path.exists(toy_path):
    toy = pickle.load(open(toy_path, "rb")) # committed tiny dataset
else:                                       # first time: subsample the source
    src = os.path.join(SIM, TP, "glm_mle_true_population.pkl")
    toy = pickle.load(open(src, "rb")).sample(n=N_ROWS, random_state=SEED).reset_index(drop=True)
    os.makedirs(TOY_DATA, exist_ok=True); pickle.dump(toy, open(toy_path, "wb"))
# the real code loads the population from its working directory:
pickle.dump(toy, open(TP + "/glm_mle_true_population.pkl", "wb"))

print("toy population:", toy.shape)
display(toy.head())
print("training rows per vital rate:",
      {n: xy(toy)[0].shape[0] for n, xy in
       [("sur",XY_sur_compu),("fec",XY_fec_compu),("flow",XY_flow_compu),
        ("grw_f",XY_grw_f_compu),("grw_nf",XY_grw_nf_compu)]})

toy population: (40, 7)


,size,sizeNext,fec,flow,surv,age,ageNext
0,NaN,0.890907,NaN,NaN,NaN,NaN,1.0
1,1.890401,2.598066,1.0,1.0,1.0,1.0,2.0
2,1.720783,NaN,0.0,NaN,0.0,5.0,NaN
3,0.511280,0.886616,0.0,NaN,1.0,1.0,2.0
4,1.483437,NaN,0.0,NaN,0.0,2.0,NaN


training rows per vital rate: {'sur': 25, 'fec': 25, 'flow': 8, 'grw_f': 7, 'grw_nf': 10}


> Corresponding files in the full analyses:
> 
> *Simulation study*
> - `simulation_case_study/s0_preprocess.ipynb` : *notebook*; simulates the two
>   ground-truth datasets `D_glm` and `D_gp` (fits GLM/GP "true" vital rate models,
>   iterates the IBM to grow a synthetic population), and fits the GLM baselines by
>   MCMC.
> - `simulation_case_study/s0_fun_base.py` – *library*; the `XY_*_compu` slicers
>   used in the cell above (`XY_sur_compu`, `XY_fec_compu`, `XY_flow_compu`,
>   `XY_grw_f_compu`, `XY_grw_nf_compu`, `XY_grw_compu`).
> 
> *Real study*
> - `real_case_study/ABC model fitting/r1_data_generating.py` – *script*; integrate
>   the nine *C. flava* yearly datasets `control_34.csv`, `control_45.csv`,
>   `control_56.csv`, `control_67.csv`, `control_78.csv`, `control_89.csv`,
>   `control_910.csv`, `control_1011.csv` and `control_1112.csv`, together with
>   `true_popu/weather_summary_2.txt`, caps age at the absorbing class, and builds
>   the training and pooled populations.
> 
> *Simulation vs real:* the simulation *vital rate models* are **size-only** (age is
> tracked and has an absorbing class at 13, but is not used as a covariate); in the
> real study the vital rates additionally depend on **age** and three **weather
> covariates**, with the absorbing "greybeard" class at 8.

---
## Step 1: a Gaussian process for each vital rate  (fitted by MCMC)

**Purpose:** Each vital rate function is assigned a **GP prior**, fitted
by **Bayesian MCMC** (using a **Hamiltonian Monte Carlo (HMC)** sampler).
Survival and fecundity use a GP with a Bernoulli likelihood (`GPMC`), the number of flowering stalks a Poisson `GPMC`, and growth a Gaussian-likelihood GP regression (`GPR`). The retained MCMC draws form the candidate pool, from which the ABC step assembles combinations of vital rate models.

Below is the *actual* fitting code (identical in shape to the real scripts, just
tiny). We show survival in full, then loop over the rest.

In [4]:
import gpflow, tensorflow as tf, tensorflow_probability as tfp
from tensorflow_probability import distributions as tfd
tf.get_logger().setLevel("ERROR") 
f64 = gpflow.utilities.to_default_float
tf.random.set_seed(SEED); np.random.seed(SEED)

def run_mcmc(model, step_size):
    "Sample the GP hyperparameters (and latent values, for GPMC) by MCMC (HMC)."
    hmc_helper = gpflow.optimizers.SamplingHelper(
        model.log_posterior_density, model.trainable_parameters)
    hmc = tfp.mcmc.HamiltonianMonteCarlo(
        target_log_prob_fn=hmc_helper.target_log_prob_fn,
        num_leapfrog_steps=N_LEAPFROG, step_size=f64(step_size))
    adaptive = tfp.mcmc.SimpleStepSizeAdaptation(
        hmc, num_adaptation_steps=int(N_BURNIN*0.7),
        target_accept_prob=f64(0.8), adaptation_rate=f64(0.1))
    @tf.function
    def run():
        return tfp.mcmc.sample_chain(
            num_results=N_DRAWS, num_burnin_steps=N_BURNIN,
            current_state=hmc_helper.current_state, kernel=adaptive,
            trace_fn=lambda _, pkr: pkr.inner_results.is_accepted)
    return run()[0]

def make_gpmc(data, likelihood, prior_scale):
    m = gpflow.models.GPMC(data=data, kernel=gpflow.kernels.RBF(), likelihood=likelihood)
    m.kernel.variance.prior = tfd.HalfNormal(f64(prior_scale))
    m.kernel.lengthscales.prior = tfd.HalfNormal(f64(prior_scale))
    return m

def make_gpr(data):
    m = gpflow.models.GPR(data=data, kernel=gpflow.kernels.RBF())
    m.kernel.lengthscales.prior = tfd.HalfNormal(f64(100.))
    m.kernel.variance.prior = tfd.HalfNormal(f64(100.))
    m.likelihood.variance.prior = tfd.HalfNormal(f64(100.))
    return m

# survival, in detail
d = XY_sur_compu(toy)
m_sur = make_gpmc((d[0], d[1]), gpflow.likelihoods.Bernoulli(), prior_scale=50.)
samples_sur = run_mcmc(m_sur, step_size=0.0001)
print("survival GP: fitted on", d[0].shape[0], "data points by MCMC (HMC) with", N_DRAWS, "draws")

survival GP: fitted on 25 data points by MCMC (HMC) with 6 draws


In [5]:
# the remaining five vital rates, same pattern
RATES = [ # name, kind, XY builder, prior scale, step size, sep/nonsep folder
    ("fec", "bern", XY_fec_compu, 100., 1.0, "sep"),
    ("flow", "poi", XY_flow_compu, 100., 0.01, "sep"),
    ("grw_f", "gpr", XY_grw_f_compu, None, 0.5, "sep"),
    ("grw_nf", "gpr", XY_grw_nf_compu,None, 0.5, "sep"),
    ("grw", "gpr", XY_grw_compu, None, 0.5, "nonsep"),
]
pickle.dump(samples_sur, open(f"{TP}/MCMC/gp/sep/samples_sur.pkl", "wb"))
for name, kind, xy, ks, step, sub in RATES:
    d = xy(toy)
    if kind == "bern": m = make_gpmc((d[0], d[1]), gpflow.likelihoods.Bernoulli(), ks)
    elif kind == "poi":  m = make_gpmc((d[0], d[1]), gpflow.likelihoods.Poisson(), ks)
    else: m = make_gpr((d[0], d[1]))
    pickle.dump(run_mcmc(m, step), open(f"{TP}/MCMC/gp/{sub}/samples_{name}.pkl", "wb"))
    print(f" fitted {name} on {d[0].shape[0]} data points")
print("all vital rate fitted by MCMC and saved.")

 fitted fec on 25 data points
 fitted flow on 8 data points
 fitted grw_f on 7 data points
 fitted grw_nf on 10 data points
 fitted grw on 17 data points
all vital rate fitted by MCMC and saved.


> **In the real code**, the GP for each vital rate is fitted by MCMC here (one
> file per vital rate; the `*2.py` twins fit the `D_gp` scenario):
> 
> *Simulation study*
> - `simulation_case_study/s0_gp_MCMC/s0_gp_MCMC_sur.py` – *script*; survival GP (Bernoulli).
> - `simulation_case_study/s0_gp_MCMC/s0_gp_MCMC_fec.py` – *script*; probability-of-reproduction GP (Bernoulli).
> - `simulation_case_study/s0_gp_MCMC/s0_gp_MCMC_flow.py` – *script*; flower-count GP (Poisson).
> - `simulation_case_study/s0_gp_MCMC/s0_gp_MCMC_grw_f.py` – *script*; breeders' growth GP (Gaussian).
> - `simulation_case_study/s0_gp_MCMC/s0_gp_MCMC_grw_nf.py` – *script*; non-breeders' growth GP (Gaussian).
> - `simulation_case_study/s0_gp_MCMC/s0_gp_MCMC_grw.py` – *script*; pooled growth GP (the `nonsep` alternative).
> 
> *Real study*
> - `real_case_study/ABC model fitting/r1_data_generating.py` – *script*; calls the
>   `mcmc_sur`, `mcmc_fec`, `mcmc_flow_poi`, `mcmc_grw_f` and `mcmc_grw_nf`
>   functions (defined in `r0_function_all.py`) to fit each vital rate's GP by
>   MCMC, using **5-D** inputs (size + age + max-temp + min-temp + precipitation).
> - `real_case_study/ss_selection/s2_simu_mcmc_sur.py`,
>   `real_case_study/ss_selection/s2_simu_mcmc_fec.py`,
>   `real_case_study/ss_selection/s2_simu_mcmc_flow_poi.py`,
>   `real_case_study/ss_selection/s2_simu_mcmc_grw_f.py`,
>   `real_case_study/ss_selection/s2_simu_mcmc_grw_nf.py` – *scripts*; per-rate GP
>   MCMC used by the summary-statistic-selection sub-package.
> 
> *Simulation vs real:* the simulation GP is **1-D** (size only); the real GP is
> **5-D** (a separate lengthscale for size, age, and each weather covariate), see Section 2.6 in the main text.

---
## Step 2: choosing summary statistics

**Purpose:** ABC compares a *simulated* population to the *observed* one, but not
row-by-row. It compares a handful of **summary statistics**, see Section 3 in the main text.

**Why standard perturbations are difficult for GPs.**  A common approach is to perturb one model parameter at a time and examine which statistics respond. For the GP models considered here, this can be unreliable because hyperparameters interact non-linearly through the kernel;
see `s1_pertur_test2.ipynb`, Appendix C.2). The paper instead measures model
*similarity* by **negative log-posterior probability (NLPP)**: it fixes a
reference model (the configuration with the lowest NLPP) and a set of
near-reference perturbed models, then, for every candidate statistic, asks *how
well does this statistic distinguish the reference from the perturbed models?* These are
scored by the **ROC-AUC** of the statistic used as a classifier. Statistics that
discriminate consistently well are kept; the rest are dropped.

First, what a summary comparison actually returns? The function below produces a five-element discrepancy vector, with one value for each statistic listed below. Smaller values indicate closer agreement between the simulated and observed populations.

For the tiny example used here, the selected discrepancy vector has five elements (see Appendix B.1 for a full description): 
 - (1) the total number of breeders across size groups (Sum of squared differences, SSD), 
 - (2) the average number of flowering stalks across size groups (SSD), 
 - (3) the total number of individuals across sizeNext groups (chi-square distance), 
 - (4) the sizeNext distribution of non-breeders (Bhattacharyya distance),
 - (5) the total number of survivors across size groups (SSD). 

In [6]:
from s0_fun_ss import list_comparisons_interested

same = list_comparisons_interested(exp_1step_data=toy, simu_1step_data=toy, truedata_style="glm")
toy_jit = toy.copy(); toy_jit["sizeNext"] = toy_jit["sizeNext"] + 1.0
diff = list_comparisons_interested(exp_1step_data=toy, simu_1step_data=toy_jit, truedata_style="glm")
print("the SELECTED summary-statistic discrepancy vector (this is what ABC scores):")
print(" observed vs observed :", np.round(np.array(same).ravel(), 3))
print(" observed vs jittered :", np.round(np.array(diff).ravel(), 3))

the SELECTED summary-statistic discrepancy vector (this is what ABC scores):
 observed vs observed : [ 0.  0.  0. -0.  0.]
 observed vs jittered : [ 0.     0.    21.786  0.718  0.   ]


The full selection workflow evaluates how well each candidate summary statistic distinguishes simulations from a reference model from those generated by nearby perturbed models. This is quantified using the AUC: higher AUC values indicate that the statistic is more sensitive to changes, whereas values close to 0.5 indicate little discriminatory ability.


In this demo, we perturb parameter values for *sizeNext*, so statistics based on the next-size distribution are expected to change.

In [7]:
from sklearn.metrics import roc_curve, auc

rng = np.random.default_rng(SEED)
def discrepancy(next_size_sd):
    d = toy.copy(); d["sizeNext"] = d["sizeNext"] + rng.normal(0, next_size_sd, len(d))
    return np.array(list_comparisons_interested(exp_1step_data=toy, simu_1step_data=d,
                                                truedata_style="glm")).ravel()

near = np.array([discrepancy(0.05) for _ in range(25)])   # barely-different  -> label 0
far  = np.array([discrepancy(1.20) for _ in range(25)])   # clearly-different -> label 1
labels = np.r_[np.zeros(25), np.ones(25)]

print("AUC of each SELECTED statistic (1.0 = perfectly separates near vs far; 0.5 = useless):")
for j in range(near.shape[1]):
    fpr, tpr, _ = roc_curve(labels, np.r_[near[:, j], far[:, j]])
    print(f"  statistic {j}: AUC = {auc(fpr, tpr):.2f}")
print("\n(We perturbed next-size, so the size-distribution statistics discriminate better. ")
print(" The real workflow runs this AUC test over the FULL candidate bank, keeping the consistently high ones.)")

AUC of each SELECTED statistic (1.0 = perfectly separates near vs far; 0.5 = useless):
  statistic 0: AUC = 0.50
  statistic 1: AUC = 0.50
  statistic 2: AUC = 1.00
  statistic 3: AUC = 1.00
  statistic 4: AUC = 0.50

(We perturbed next-size, so the size-distribution statistics discriminate better. 
 The real workflow runs this AUC test over the FULL candidate bank, keeping the consistently high ones.)


> **In the real code**, summary statistics are defined and selected here:
> 
> *Simulation study*
> - `simulation_case_study/s0_fun_ss.py` – *library*; `list_comparisons` (the full
  > candidate bank of statistics) and `list_comparisons_interested` (the selected
  > subset, with separate `glm`/`gp` branches, the output the ABC step consumes).
> - `simulation_case_study/s0_class_GP_IPM.py` – *library*; the `Perted_IPM` class
  > that defines the NLPP-optimum reference model (`opt_comp`) and the near-optimum
  > ensemble (`whether_around_opt_comp`), and runs the paired simulations. The
  > ROC-AUC scoring itself is done in the `s1_ss_selection` notebooks below.
> - `simulation_case_study/s1_ss_selection/s1_ss_sur.ipynb`,
  > `simulation_case_study/s1_ss_selection/s1_ss_fec.ipynb`,
  > `simulation_case_study/s1_ss_selection/s1_ss_flow.ipynb`,
  > `simulation_case_study/s1_ss_selection/s1_ss_grw_f.ipynb`,
  > `simulation_case_study/s1_ss_selection/s1_ss_grw_nf.ipynb`,
  > `simulation_case_study/s1_ss_selection/s1_ss_grw.ipynb` – *notebooks*; the
  > per vital rate AUC selection (each has a `2`-suffixed twin for `D_gp`). The
  > chosen subset is then hard-coded into `list_comparisons_interested`.
> - `simulation_case_study/s1_pertur_test2.ipynb` – *notebook*; the empirical check illustrating limitations of standard one-at-a-time hyperparameter perturbations for the GP models (Appendix C.2). 
> 
> *Real study*
> - `real_case_study/ss_selection/s0_fun_teststats.py` – *library*; candidate statistics.
> - `real_case_study/ss_selection/s0_initial.py` – *library*; the candidate list
  > (`col_names`, 76 candidates for the real study).
> - `real_case_study/ss_selection/s3_explore_sur.ipynb`,
  > `real_case_study/ss_selection/s3_explore_fec.ipynb`,
  > `real_case_study/ss_selection/s3_explore_flow_poi.ipynb`,
  > `real_case_study/ss_selection/s3_explore_grw_f.ipynb`,
  > `real_case_study/ss_selection/s3_explore_grw_nf.ipynb` – *notebooks*; the
  > AUC-based selection workflow.
> - `real_case_study/ABC model fitting/test_stats.py` – *library*; summary-statistic
  > helpers used by the real ABC, with the selected set wired into `r0_function_all.py`.
> 
> *Simulation vs real:* the simulation study chooses from 31 candidate statistics; the real study chooses from 76 candidate statistics.

--- 
## Step 3: caching the GP posteriors

**Purpose:** ABC will assemble *thousands* of IPMs, and recomputing each GP posterior
every time would be far too slow. So for every stored MCMC draw we precompute and
pickle the GP's Cholesky/posterior **cache**. A built-in check confirms the
cached prediction reproduces the uncached one to within floating-point tolerance. 

First, we assemble the `ipmmcmc_whole` object (the ABC engine) with the GP
models + their MCMC draws attached, which is exactly as the real cache/ABC drivers do.

This step does not fit new models or generate new posterior samples; it only precomputes 
and stores the GP prediction caches required by later steps. The number of samples in this demo is
small, so running it locally is fine. In the real practice, the number of samples
will be much larger, so we recommend running this step on a compute server or HPC environment.

In [8]:
from s0_class_ABCPMC import ipmmcmc_whole
from s0_fun_IBMs import popu_structure, GPMC_posterior, IBM_1step_gp, IBM_1step_gp_cache

def build_whole(popu):
    "Attach the GP models + MCMC draws to an ipmmcmc_whole (as s1_cache_produce.py / s2_ABCSMC_sep.py do)."
    wy = ipmmcmc_whole(popu_data=popu, grw_setting="sep", z0=popu_structure(popu),
                       truedata_style="glm", alpha=ALPHA, beta=BETA, recruit_p=RECRUIT_P)
    def gpmc(xy, ks, lik):
        dd = xy(popu); m = make_gpmc((dd[0], dd[1]), lik, ks)
        return m, gpflow.optimizers.SamplingHelper(m.log_posterior_density, m.trainable_parameters)
    def gpr(xy):
        dd = xy(popu); m = make_gpr((dd[0], dd[1]))
        return m, gpflow.optimizers.SamplingHelper(m.log_posterior_density, m.trainable_parameters)
    wy.m_fec_new, wy.hmc_helper_fec = gpmc(XY_fec_compu, 100., gpflow.likelihoods.Bernoulli())
    wy.m_flow_poi_new,wy.hmc_helper_flow = gpmc(XY_flow_compu, 100., gpflow.likelihoods.Poisson())
    wy.m_sur_new, wy.hmc_helper_sur = gpmc(XY_sur_compu, 50., gpflow.likelihoods.Bernoulli())
    wy.m_grw_f_new, wy.hmc_helper_grw_f = gpr(XY_grw_f_compu)
    wy.m_grw_nf_new, wy.hmc_helper_grw_nf = gpr(XY_grw_nf_compu)
    wy.m_grw_new, wy.hmc_helper_grw = gpr(XY_grw_compu)
    L = lambda p: pickle.load(open(f"{TP}/MCMC/gp/{p}", "rb"))
    wy.samples_sur=L("sep/samples_sur.pkl"); wy.samples_fec=L("sep/samples_fec.pkl")
    wy.samples_flow=L("sep/samples_flow.pkl"); wy.samples_grw_f=L("sep/samples_grw_f.pkl")
    wy.samples_grw_nf=L("sep/samples_grw_nf.pkl"); wy.samples_grw=L("nonsep/samples_grw.pkl")
    wy.rep = 1
    return wy

wy = build_whole(toy)

LM = "true_popu/mcmc/Lm/glm"
with contextlib.redirect_stdout(io.StringIO()): # silence the internal "cache only" message
    for i in range(N_DRAWS):
        wy.GPmodels_givenindex_cache([i, i, i, i, i, i])
        pickle.dump(GPMC_posterior(wy.m_sur_new).cache, open(f"{LM}/sur/{i}", "wb"))
        pickle.dump(wy.m_grw_f_new.posterior().cache, open(f"{LM}/grw_f/{i}", "wb"))
        pickle.dump(wy.m_grw_nf_new.posterior().cache, open(f"{LM}/grw_nf/{i}", "wb"))
        pickle.dump(GPMC_posterior(wy.m_flow_poi_new).cache, open(f"{LM}/flow/{i}", "wb"))
        pickle.dump(GPMC_posterior(wy.m_fec_new).cache, open(f"{LM}/fec/{i}", "wb"))
        pickle.dump(wy.m_grw_new.posterior().cache, open(f"{LM}/grw/{i}", "wb"))

    # correctness: the cached IBM step must reproduce the uncached one (same seed)
    wy.GPmodels_givenindex_cache([0, 0, 0, 0, 0, 0])
    for k, r in [("m_sur","sur"),("m_grw_f","grw_f"),("m_grw_nf","grw_nf"),("m_fec","fec"),("m_flow_poi","flow")]:
        getattr(wy, k + "_new").cache = pickle.load(open(f"{LM}/{r}/0", "rb"))
    models0 = {"m_sur":wy.m_sur_new,"m_grw_f":wy.m_grw_f_new,"m_grw_nf":wy.m_grw_nf_new,
               "m_fec":wy.m_fec_new,"m_flow_poi":wy.m_flow_poi_new,
               "alpha":ALPHA,"beta":BETA,"recruit_p":RECRUIT_P}
    np.random.seed(4567); a = IBM_1step_gp(zt=wy.z0[0], age=wy.z0[1], models=models0)
    np.random.seed(4567); b = IBM_1step_gp_cache(zt=wy.z0[0], age=wy.z0[1], models=models0)
    np.testing.assert_allclose(a, b)
print(f"cached {N_DRAWS} draws x 6 GP models (including the pooled-growth).")

cached 6 draws x 6 GP models (including the pooled-growth).


> **In the real code**, caching is done here:
> 
> *Simulation study*
> - `simulation_case_study/s1_cache_produce.py` – *script*; precomputes and pickles
  > the GP posterior/Cholesky cache for every MCMC draw into
  > `true_popu/mcmc/Lm/glm/<rate>/<idx>`, and validates cached == uncached.
  > (`s1_cache_produce2.py` = the `D_gp` scenario.)
> - `simulation_case_study/s0_fun_IBMs.py` – *library*; the cache helpers
  > `GPMC_posterior`, `predict_y_loaded_cache`, `predict_f_loaded_cache`.
> 
> *Real study*
> - `real_case_study/ABC model fitting/r2_cache_producing_and_testing.py` –
  > *script*; the same caching + cached-vs-uncached assertion for the real 5-D models.

---
## Step 4: the simulation engine (the individual-based model)

**Purpose:** ABC methods relys on simulations. The simulator in this study is 
an individual-based model (IBM): it takes the current population and the fitted 
vital rate functions, and advances every individual one year forward, drawing 
each life-cycle event at random. This is where demographic stochasticity enters.

The order of events follows the census structure, so **reproduction is simulated
before survival**:

1. **Flowering.** For each individual of size $z$, draw $\mathrm{Fl}\sim\mathrm{Bern}(p_f(z))$,
   whether it flowers this year.
2. **Flowering stalks.** For each flowering individual, draw the number of stalks
   as $\mathrm{Pois}(n_f(z))+1$; the shift by 1 imposes that a flowering
   individual will have at least 1 flowering stalk.
3. **Recruits.** Draw the number of new recruits over all stalks
   produced in the population, $\mathrm{Binom}(\sum \mathrm{stalks},\, r_{est})$.
4. **Recruit size.** Give each recruit a size from the recruit size distribution
   $\mathrm{Gamma}(\alpha,\beta)$, which is independent of parents.
5. **Survival.** For each individual, draw $\mathrm{Surv}\sim\mathrm{Bern}(s(z))$.
6. **Growth.** Each survivor grows. Growth is conditioned on flowering status: 
   survivors that flowered grow under $G_f$; survivors that did not flower grow under $G_{nf}$.
   The next size is drawn using the predictive mean and variance returned by the fitted growth GP.
7. **Age.** Survivors age by one year, up to an absorbing final age class;
   recruits enter at age one.

Let's run 1 such step and see what comes.

In [9]:
from s0_fun_IBMs import IBM_1step_gp_cache, popu_structure

# assemble one candidate model (one MCMC draw per vital rate) from the caches
models_one = wy.new_model(np.array([0, 1, 2, 3, 4]))

np.random.seed(SEED)
simulated = IBM_1step_gp_cache(zt=wy.z0[0], age=wy.z0[1], models=models_one)

print("starting population:", len(wy.z0[0]), "individuals")
print("simulated one year forward:", len(simulated), "rows")
print()
print("rows with sizeNext but no size  = recruits born this year:",
      int((simulated['size'].isna() & simulated['sizeNext'].notna()).sum()))
print("survivors (surv == 1):",
      int((simulated['surv'] == 1).sum()))
print("individuals that flowered (fec == 1):",
      int((simulated['fec'] == 1).sum()))
display(simulated.head())

starting population: 32 individuals
simulated one year forward: 35 rows

rows with sizeNext but no size  = recruits born this year: 3
survivors (surv == 1): 13
individuals that flowered (fec == 1): 13


,size,sizeNext,fec,flow,surv,age,ageNext
0,0.890907,4.248958,0.0,NaN,1.0,1.0,2.0
1,2.598066,NaN,0.0,NaN,0.0,2.0,NaN
2,0.886616,3.361944,0.0,NaN,1.0,2.0,3.0
3,0.283130,NaN,0.0,NaN,0.0,1.0,NaN
4,1.625651,1.700731,0.0,NaN,1.0,1.0,2.0


The output has exactly the same columns as the observed data (Step 0), which is
what makes the two datasets comparable. Note the
two ways a row can appear: an established individual carries both `size` and
`surv`, whereas a recruit born this year has no `size` and enters with
`ageNext` equal to one.

Because the draws are random, running the cell again with a different seed gives
a different population from the *same* model. 

> **In the real code**, the IBM is implemented here:
>
> *Simulation study*
> - `simulation_case_study/s0_fun_IBMs.py` – all IBM variants. `IBM_1step_gp` is the
>   version used with GP vital rates, `IBM_1step_gp_cache` the numerically identical
>   version that reads the precomputed GP posteriors and is the one ABC calls,
>   `IBM_1step_glm_mle` the version used to generate the `D_glm` data set (`D_gp` was
>   generated with `IBM_1step_gp`), and
>   `IBM_1step_glm` the version driven by GLM posterior draws. The `_same` variants
>   are for the pooled-growth (`nonsep`) setting. `popu_structure` extracts the
>   starting population for the next step.
>
> *Real study*
> - `real_case_study/ABC model fitting/r0_function_all.py` – `IBM_1step` and its cached
>   counterpart `IBM_1step_cashe`, following the same seven steps but with vital rates
>   that also depend on age and on the three weather covariates, and with the absorbing
>   age class of the age-size structured model.
> - `real_case_study/ss_selection/s0_fun_IBM.py` – the IBM used by the
>   summary statistic selection workflow.

--- 
## Step 5: Population-level calibration with ABC

**Purpose:** We use ABC to incorporate population-level information when an explicit 
tractable likelihood linking the fitted vital rate models to the chosen population summaries is not available. 
Instead of evaluating a likelihood, ABC
*simulates* a population from a candidate model and keeps the candidate only if
its summary statistics are close enough to the observed ones.

**The particle representation:** In this codebase, an ABC **particle is a tuple of integer
indices**, one per vital rate, each pointing into that rate's MCMC draw chain. So 
a particle is formed by combining one retained posterior draw from each independently 
fitted vital rate model, and "perturbing a vital rate" just means swapping one index 
(Algorithm 1 in Appendix A in the paper). 

For each particle the sampler:
1. assembles the IPM from the cached GP posteriors for those indices,
2. runs IBM step (Step 4) to generate a simulated dataset,
3. reduces the result to the selected summary statistics (Step 2) and scores its
   distance to the observed data,
4. **accepts** it if the distance is below the current tolerance, then reweights.

The first round is a special case: every candidate is kept, and the round only
calibrates the MAD scaling and sets the first tolerance from the given quantile. 
Filtering begins at the second round. 

ABC therefore provides a population-informed calibration step that favours vital rate combinations 
producing simulated summaries closer to the observed ones. Across ABC rounds the tolerance 
shrinks (so the accepted set concentrates), and the kept particles are reweighted.

Below, we let a **single particle** make one pass: propose, simulate, score.

Note that a particle carries **five** indices although six GPs were fitted: the toy
run uses the `sep` setting, so the pooled `grw` GP is only the unused `nonsep`
alternative, and recruitment is not a GP.

In [10]:
wy.rep = 1
demo_idx = np.array([0, 1, 2, 3, 4])           # example index, one MCMC-draw index per vital rate
_, summ = wy.IPM_whole_givenindex(demo_idx)    # assemble IPM -> run 1 IBM step -> compare to observed
print("particle", demo_idx.tolist(), "[sur, grw_f, grw_nf, fec, flow]")
print(" summary-statistic discrepancy vs observed:", np.round(np.array(summ).ravel(), 2))
print(" ABC turns this into a single (normalised) distance and accepts the")
print(" particle if it falls below the current tolerance.")

particle [0, 1, 2, 3, 4] [sur, grw_f, grw_nf, fec, flow]
 summary-statistic discrepancy vs observed: [13.   87.5   7.25  0.49 14.  ]
 ABC turns this into a single (normalised) distance and accepts the
 particle if it falls below the current tolerance.


The sampler repeats this procedure for **many** particles across rounds with progressively tighter tolerances. 

The full implementation evaluates large candidate batches in parallel; this toy example uses a small serial loop and restricts perturbations to the available cache indices; no source file is changed.

In [11]:
from s0_fun_ss import list_comparisons_interested  # (used inside IPM_whole_givenindex)

def _serial(self, total_samples, given_theta=False, given_weight=False, random=True):
    out = []
    for _ in range(int(min(total_samples, self.toy_batch))):
        if given_theta and given_weight and random: out.append(self.random_IPM_ABC_weight())
        elif given_theta and given_weight and not random: out.append(self.not_random_IPM_ABC_weight())
        else: out.append(self.random_IPM_whole())
    return out

def _perturb(self): # perturb within the toy's cache range
    idx = self.p_index[np.random.choice(self.p_index.shape[0], p=self.weight)].copy()
    idx[np.random.randint(0, 5)] = np.random.randint(0, self.n_draws)
    return self.IPM_whole_givenindex(idx)

wy.toy_batch = ABC_BATCH; wy.n_draws = N_DRAWS
wy.random_IPM_whole_para = types.MethodType(_serial, wy)
wy.random_IPM_ABC_weight = types.MethodType(_perturb, wy)

np.random.seed(SEED)
p_index, threshold = wy.ABC_SMC(quantiles=np.array(ABC_QUANTILES),
                                n_particles=np.array(ABC_NPARTICLES), details=True)
print("\nABC posterior:", p_index.shape, "particles (each = 5 vital rate indices)")
print("example particles [sur, grw_f, grw_nf, fec, flow]:")
print(np.unique(p_index, axis=0)[:5])

c=0 11:33:26
Total: 60
Left: 60
Unique: 60 

c=1 11:33:28
  11:33:32 Required: 30, Now: 72
Left: 72
Unique: 71 


ABC posterior: (72, 5) particles (each = 5 vital rate indices)
example particles [sur, grw_f, grw_nf, fec, flow]:
[[0 0 5 1 1]
 [0 2 0 3 4]
 [0 2 3 2 0]
 [0 2 4 2 3]
 [0 3 0 5 1]]


> **In the real code**, the ABC sampler is here:
> 
> *Simulation study*
> - `simulation_case_study/s0_class_ABCPMC.py` – *library*; the ABC engine
  > `ipmmcmc_whole` (`ABC_SMC` = the ABC loop; `new_model` = load caches + assign
  > draws; `IPM_whole_givenindex` = simulate one IBM step + score).
> - `simulation_case_study/s2_ABCSMC_sep.py` – *script*; builds `ipmmcmc_whole`,
  > attaches the models, and runs `ABC_SMC` → writes `ABC_details/glmsep/`.
  > (`s2_ABCSMC_sep2.py` = the `D_gp` scenario.)
> 
> *Real study*: the core functions live in an **import chain**; the file you
> `import` (`r0_function`) is a one-line shim, so read `r0_function_all.py`:
> - `real_case_study/ABC model fitting/r0_function_all.py` – *library*; the
  > **definition root** (the `XY_*` builders, `IBM_1step`, `list_comparisons_interested`,
  > cache helpers).
> - `real_case_study/ABC model fitting/r0_function_seperate.py` – *library*; imports
  > `_all`, adds the per-year `ipmmcmc`.
> - `real_case_study/ABC model fitting/r0_function_whole.py` – *library*; imports
  > `_seperate`, adds `ipmmcmc_whole` + `ABC_SMC`.
> - `real_case_study/ABC model fitting/r0_function.py` – *library*; a one-line shim
  > (`from r0_function_whole import *`).
> - `real_case_study/ABC model fitting/r4_ABCSMC.py` – *script*; the real ABC
  > driver (11 tolerance rounds).
> 
> *Simulation vs real:* the simulation ABC matches one transition over ~7 rounds;
> the real ABC simulates all training years and runs ~11 rounds.
> 
> Folder `ABC_details/` stores the accepted index tuples and weights, together with
> the per-round diagnostics (`threshold*`, `mad*`, `accepted*`, `particles_summary*`,
> `r_s_mean*`, `r_s_median*`), but **no fitted parameters**. To recover parameters or
> predictions you combine the indices with the saved MCMC draws and the GP caches
> from Steps 1 and 3.

---
## Step 6: what you do with the ABC output

**Purpose:** In this code base, the ABC output is **a weighted set of index tuples**. You use it by
assembling an IPM (or running an IBM forecast) for each accepted particle and
reading off population-level quantities, for example, the growth rate λ (dominant eigenvalue
of `P+F`), the stable stage distribution, reproductive value, or multi-year
forecasts with prediction intervals. **This is where the paper's figures and the
GLM / GP / ABC_GP comparison are produced.**

As a one-line illustration, we take a single accepted particle, assemble its IPM
from the caches, and read off its λ. 

In [12]:
from s0_fun_IPMs import IPM_listk_gpcache, kernel_setting

particle = p_index[0]  # one accepted particle (5 indices)
models = wy.new_model(particle) # loads its GP caches + assigns its MCMC draws
mesh = kernel_setting(12, 0, 5)  # 12-point mesh over log-size [0,5]
P, F = IPM_listk_gpcache(mesh, models, "sep")  # assemble the discretised IPM
eigvals = np.linalg.eigvals(P + F)
lam = eigvals[np.argmax(np.abs(eigvals))].real # dominant eigenvalue (lambda)
print("particle", particle.tolist(), "-> assembled IPM -> lambda =", round(lam, 3), "  (illustrative only)")

particle [3, 0, 5, 0, 4] -> assembled IPM -> lambda = 0.697   (illustrative only)


> **In the real code**, the post-ABC processing (where the actual results,
> forecasts and figures come from) is here:
> 
> *Simulation study*
> - `simulation_case_study/s2_IPM_compare.ipynb` – *notebook*; builds IPM kernels
  > per method (GLM / GP / ABC_GP), computes λ (MSE/bias vs truth), the stable stage
  > distribution (KL) and reproductive value (cosine), and the λ comparison figure.
  > (`s2_IPM_compare2.ipynb` = the `D_gp` scenario.)
> - `simulation_case_study/s3_marginal.ipynb` – *notebook*; marginal vital rate /
  > posterior plots. (`s3_marginal2.ipynb` = `D_gp`.)
> - `simulation_case_study/s5_efficiency.ipynb` – *notebook*; ABC efficiency
  > diagnostics (ESS, acceptance, degeneracy). (`s5_efficiency2.ipynb` = `D_gp`.)
> - `simulation_case_study/s4_glm_recheck.ipynb` – *notebook*; re-checks the GLM
  > baseline on a larger population.
> 
> *Real study*
> - `real_case_study/ABC model fitting/r5_lambda_abc.py` – *script*; **ABC_GP**
  > long-term population forecast.
> - `real_case_study/ABC model fitting/r5_lambda_abc_z.py` – *script*; **ABC_GP**
  > one-step-ahead forecast (the `_z` suffix = one-step).
> - `real_case_study/ABC model fitting/r5_lambda_random.py` / `r5_lambda_random_z.py`
  > – *scripts*; the **GP** baseline (independent draws), long-term / one-step.
> - `real_case_study/ABC model fitting/r5_lambda_glm.ipynb` – *notebook*; the
  > **GLM** baseline forecast.
> - `real_case_study/ABC model fitting/r6_ acceptance_compare.ipynb` /
  > `r6_ acceptance_compare_testyears.ipynb` – *notebooks*; population trajectories,
  > 95% prediction intervals, per-year log-λ, and marginal hyperparameter posteriors.
> - `real_case_study/ABC model fitting/r7_efficiency.ipynb` – *notebook*; ABC
  > efficiency (ESS = 1/Σw², neighbour/degeneracy counts).
> 
> The three methods differ in both the vital rate models and how posterior draws are assembled: 
> - GLM uses Bayesian GLM vital rate models; 
> - GP uses independently assembled posterior draws from the separately fitted GP models; 
> - ABC_GP uses the same GP posterior draws but reweights/selects their combinations through the ABC step (Step 5).

---
## Recap

The workflow is summarised below:

| step | what it does | key files (simulation / real) |
|---|---|---|
| 0 data | one row per individual per transition | `s0_preprocess.ipynb` / `r1_data_generating.py` |
| 1 GP per vital rate | GP model fitted by MCMC (HMC) | `s0_gp_MCMC/s0_gp_MCMC_*.py` / `r1_data_generating.py` + `ss_selection/s2_simu_mcmc_*.py` |
| 2 summary statistics | pick the few numbers ABC matches (NLPP + AUC) | `s0_fun_ss.py` + `s1_ss_selection/s1_ss_*.ipynb` / `ss_selection/s3_explore_*.ipynb` |
| 3 cache | precompute GP posteriors | `s1_cache_produce.py` / `r2_cache_producing_and_testing.py` |
| 4 IBM | simulate the population one year forward, the engine ABC runs | `s0_fun_IBMs.py` / `r0_function_all.py` |
| 5 ABC | calibrate/reweight vital rate combinations using population-level summaries | `s0_class_ABCPMC.py` + `s2_ABCSMC_sep.py` / `r0_function_whole.py` + `r4_ABCSMC.py` |
| 6 use the output | λ, stable dist, forecasts, GLM/GP/ABC_GP comparison | `s2_IPM_compare.ipynb` / `r5_lambda_*`, `r6_ acceptance_compare*.ipynb`, `r7_efficiency.ipynb` |

**Reminder:** numerical results from this toy example (~40 rows and ~6 posterior draws) are not intended for inference or performance evaluation. The notebook demonstrates the code path only.

For a file-by-file overview of the whole repository, see the `README.md` at the repository root.